# TP2bis — Réduction de Dimension — Corrigé

Ce corrigé couvre toutes les parties du TP avec des explications détaillées.

In [ ]:
# Installation des dépendances
# !pip install scikit-learn matplotlib numpy pandas umap-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Partie 1 — Motivation et fléau de la dimension

In [ ]:
# 1.1 Fléau de la dimensionnalité
from sklearn.neighbors import NearestNeighbors

def mean_nn_distance(n_samples, n_dims):
    """Calcule la distance moyenne au plus proche voisin."""
    np.random.seed(42)
    X = np.random.uniform(0, 1, size=(n_samples, n_dims))
    nn = NearestNeighbors(n_neighbors=2)
    nn.fit(X)
    distances, _ = nn.kneighbors(X)
    return distances[:, 1].mean()  # Distance au 1er voisin (pas soi-même)

dimensions = [2, 3, 5, 10, 20, 50, 100]
n_samples = 100

distances = [mean_nn_distance(n_samples, d) for d in dimensions]

plt.figure(figsize=(10, 5))
plt.plot(dimensions, distances, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Nombre de dimensions', fontsize=12)
plt.ylabel('Distance moyenne au plus proche voisin', fontsize=12)
plt.title('Fléau de la dimensionnalité : les points s\'éloignent en haute dimension', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("=== Distances moyennes ===")
for d, dist in zip(dimensions, distances):
    print(f"  {d:3d}D : {dist:.4f}")

In [ ]:
# 1.2 Pourquoi réduire ?
print("=== Motivations de la réduction de dimension ===")
print()
print("1. VISUALISATION : projeter en 2D/3D pour explorer")
print("2. DÉBRUITAGE : éliminer les dimensions non informatives")
print("3. COMPRESSION : réduire la mémoire nécessaire")
print("4. PRÉ-TRAITEMENT ML : améliorer vitesse et performance")
print("5. RÉGULARISATION : éviter le surapprentissage")

## Partie 2 — Analyse en Composantes Principales (PCA)

In [ ]:
# Chargement du dataset Iris
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names

# Standardisation
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

print(f"Shape Iris : {X_iris.shape}")
print(f"Features : {feature_names}")

In [ ]:
# 2.2 PCA pas à pas (sans scikit-learn)
print("=== PCA pas à pas ===")
print()

# 1. Centrer (déjà fait par StandardScaler)
X_centered = X_iris_scaled

# 2. Matrice de covariance
cov_matrix = np.cov(X_centered.T)
print("Matrice de covariance (4x4) :")
print(np.round(cov_matrix, 3))

# 3. Décomposition en valeurs propres
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# 4. Tri décroissant
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx].real
eigenvectors = eigenvectors[:, idx].real

print(f"\nValeurs propres : {np.round(eigenvalues, 3)}")
print(f"Variance expliquée : {np.round(eigenvalues / eigenvalues.sum() * 100, 1)}%")

# 5. Projection
k = 2
X_pca_manual = X_centered @ eigenvectors[:, :k]
print(f"\nShape après projection : {X_pca_manual.shape}")

In [ ]:
# 2.3 PCA avec scikit-learn
from sklearn.decomposition import PCA

pca_iris = PCA(n_components=2)
X_pca_sklearn = pca_iris.fit_transform(X_iris_scaled)

# Vérification : résultats similaires
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_pca_manual[:, 0], X_pca_manual[:, 1], c=y_iris, cmap='viridis', s=50)
axes[0].set_title('PCA manuel (NumPy)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

axes[1].scatter(X_pca_sklearn[:, 0], X_pca_sklearn[:, 1], c=y_iris, cmap='viridis', s=50)
axes[1].set_title('PCA scikit-learn')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

In [ ]:
# 2.4 Variance expliquée
pca_full = PCA()
pca_full.fit(X_iris_scaled)

var_ratio = pca_full.explained_variance_ratio_
cumsum_var = np.cumsum(var_ratio)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(var_ratio)+1), var_ratio, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Composante')
axes[0].set_ylabel('Variance expliquée')
axes[0].set_title('Scree Plot')
axes[0].set_xticks(range(1, len(var_ratio)+1))

# Variance cumulée
axes[1].plot(range(1, len(cumsum_var)+1), cumsum_var, 'b-o', linewidth=2)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% variance')
axes[1].set_xlabel('Nombre de composantes')
axes[1].set_ylabel('Variance cumulée')
axes[1].set_title('Variance cumulée')
axes[1].set_xticks(range(1, len(cumsum_var)+1))
axes[1].legend()

plt.tight_layout()
plt.show()

# Combien pour 95% ?
n_95 = np.argmax(cumsum_var >= 0.95) + 1
print(f"Composantes pour 95% de variance : {n_95}")
print(f"Variance par composante : {np.round(var_ratio * 100, 1)}%")

In [ ]:
# 2.5 Interprétation des composantes
print("=== Poids des features dans les composantes ===")
components_df = pd.DataFrame(
    pca_full.components_[:2],
    columns=feature_names,
    index=['PC1', 'PC2']
)
display(components_df.round(3))

# Biplot
fig, ax = plt.subplots(figsize=(10, 8))

# Points
scatter = ax.scatter(X_pca_sklearn[:, 0], X_pca_sklearn[:, 1], c=y_iris, cmap='viridis', s=50, alpha=0.7)

# Vecteurs des features
for i, fname in enumerate(feature_names):
    ax.arrow(0, 0, pca_full.components_[0, i]*3, pca_full.components_[1, i]*3,
             head_width=0.1, head_length=0.05, fc='red', ec='red')
    ax.text(pca_full.components_[0, i]*3.3, pca_full.components_[1, i]*3.3, 
            fname.replace(' (cm)', ''), fontsize=10, color='red')

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Biplot PCA — Iris')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.colorbar(scatter, label='Espèce')
plt.tight_layout()
plt.show()

## Partie 3 — Variantes de PCA

In [ ]:
# 3.1 Kernel PCA
from sklearn.decomposition import KernelPCA
from sklearn.datasets import make_circles

# Dataset non linéairement séparable
X_circles, y_circles = make_circles(n_samples=500, factor=0.3, noise=0.05, random_state=42)

# Comparaison PCA vs Kernel PCA
pca_linear = PCA(n_components=2)
X_pca_circles = pca_linear.fit_transform(X_circles)

kpca_rbf = KernelPCA(n_components=2, kernel='rbf', gamma=15)
X_kpca_circles = kpca_rbf.fit_transform(X_circles)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', s=30)
axes[0].set_title('Données originales')

axes[1].scatter(X_pca_circles[:, 0], X_pca_circles[:, 1], c=y_circles, cmap='viridis', s=30)
axes[1].set_title('PCA linéaire (échec)')

axes[2].scatter(X_kpca_circles[:, 0], X_kpca_circles[:, 1], c=y_circles, cmap='viridis', s=30)
axes[2].set_title('Kernel PCA (RBF)')

plt.suptitle('PCA vs Kernel PCA sur make_circles', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Sparse PCA
from sklearn.decomposition import SparsePCA

spca = SparsePCA(n_components=2, alpha=0.5, random_state=42)
X_spca = spca.fit_transform(X_iris_scaled)

print("=== Comparaison PCA vs Sparse PCA ===")
print("\nPCA standard (composantes denses) :")
print(np.round(pca_full.components_[:2], 3))

print("\nSparse PCA (composantes parcimonieuses) :")
print(np.round(spca.components_, 3))

print("\n→ Sparse PCA a des coefficients nuls, plus interprétable !")

## Partie 4 — t-SNE

In [ ]:
# Dataset Digits
from sklearn.datasets import load_digits

digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Shape Digits : {X_digits.shape} (images 8x8 = 64 pixels)")

# Visualiser quelques chiffres
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Label: {y_digits[i]}')
    ax.axis('off')
plt.suptitle('Exemples de chiffres (8x8 pixels)')
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Application t-SNE
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X_digits)

plt.figure(figsize=(12, 10))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits, cmap='tab10', s=30, alpha=0.8)
plt.colorbar(scatter, label='Chiffre')
plt.title('t-SNE sur Digits (perplexity=30)', fontsize=14)
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Effet de perplexity
perplexities = [5, 30, 50, 100]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, perp in zip(axes.flat, perplexities):
    tsne_temp = TSNE(n_components=2, perplexity=perp, n_iter=1000, random_state=42)
    X_temp = tsne_temp.fit_transform(X_digits)
    scatter = ax.scatter(X_temp[:, 0], X_temp[:, 1], c=y_digits, cmap='tab10', s=20, alpha=0.8)
    ax.set_title(f'perplexity = {perp}')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Effet de perplexity sur t-SNE', fontsize=14)
plt.tight_layout()
plt.show()

print("perplexity bas (5) : clusters très serrés, structure locale")
print("perplexity haut (100) : clusters plus diffus, structure globale")

## Partie 5 — UMAP

In [ ]:
import umap

# 5.3 Application UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X_digits)

# Comparaison t-SNE vs UMAP
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

scatter1 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_digits, cmap='tab10', s=30, alpha=0.8)
axes[0].set_title('t-SNE (perplexity=30)', fontsize=13)
axes[0].set_xticks([])
axes[0].set_yticks([])

scatter2 = axes[1].scatter(X_umap[:, 0], X_umap[:, 1], c=y_digits, cmap='tab10', s=30, alpha=0.8)
axes[1].set_title('UMAP (n_neighbors=15)', fontsize=13)
axes[1].set_xticks([])
axes[1].set_yticks([])

plt.colorbar(scatter2, ax=axes, label='Chiffre', shrink=0.8)
plt.suptitle('Comparaison t-SNE vs UMAP sur Digits', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Effet des hyperparamètres UMAP
n_neighbors_list = [5, 15, 50, 200]
min_dist_list = [0.0, 0.1, 0.5, 0.99]

# Varier n_neighbors
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for ax, nn in zip(axes[0], n_neighbors_list):
    reducer_temp = umap.UMAP(n_neighbors=nn, min_dist=0.1, random_state=42)
    X_temp = reducer_temp.fit_transform(X_digits)
    ax.scatter(X_temp[:, 0], X_temp[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
    ax.set_title(f'n_neighbors={nn}')
    ax.set_xticks([])
    ax.set_yticks([])

# Varier min_dist
for ax, md in zip(axes[1], min_dist_list):
    reducer_temp = umap.UMAP(n_neighbors=15, min_dist=md, random_state=42)
    X_temp = reducer_temp.fit_transform(X_digits)
    ax.scatter(X_temp[:, 0], X_temp[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
    ax.set_title(f'min_dist={md}')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Effet des hyperparamètres UMAP', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 5.5 UMAP supervisé
reducer_supervised = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap_supervised = reducer_supervised.fit_transform(X_digits, y=y_digits)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_umap[:, 0], X_umap[:, 1], c=y_digits, cmap='tab10', s=20, alpha=0.7)
axes[0].set_title('UMAP non supervisé')
axes[0].set_xticks([])
axes[0].set_yticks([])

axes[1].scatter(X_umap_supervised[:, 0], X_umap_supervised[:, 1], c=y_digits, cmap='tab10', s=20, alpha=0.7)
axes[1].set_title('UMAP supervisé')
axes[1].set_xticks([])
axes[1].set_yticks([])

plt.suptitle('UMAP : non supervisé vs supervisé', fontsize=14)
plt.tight_layout()
plt.show()

print("UMAP supervisé utilise les labels pour mieux séparer les classes")

## Partie 6 — Autres méthodes de Manifold Learning

In [ ]:
# Dataset Swiss Roll
from sklearn.datasets import make_swiss_roll

X_swiss, color_swiss = make_swiss_roll(n_samples=1500, noise=0.1, random_state=42)

# Visualisation 3D
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2], c=color_swiss, cmap='viridis', s=10)
ax.set_title('Swiss Roll en 3D')
plt.tight_layout()
plt.show()

In [ ]:
# 6.1-6.3 Comparaison des méthodes
from sklearn.manifold import Isomap, LocallyLinearEmbedding, MDS

methods = {
    'PCA': PCA(n_components=2),
    'Isomap': Isomap(n_components=2, n_neighbors=10),
    'LLE': LocallyLinearEmbedding(n_components=2, n_neighbors=10),
    't-SNE': TSNE(n_components=2, perplexity=30, random_state=42),
    'UMAP': umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flat

for ax, (name, method) in zip(axes, methods.items()):
    X_reduced = method.fit_transform(X_swiss)
    ax.scatter(X_reduced[:, 0], X_reduced[:, 1], c=color_swiss, cmap='viridis', s=10)
    ax.set_title(name, fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])

# Masquer le 6ème subplot
axes[5].axis('off')

plt.suptitle('Déroulement du Swiss Roll par différentes méthodes', fontsize=14)
plt.tight_layout()
plt.show()

print("Isomap et LLE 'déroulent' correctement la variété !")
print("PCA échoue car c'est une méthode linéaire.")

## Partie 7 — Analyse Discriminante Linéaire (LDA)

In [ ]:
# 7.2-7.3 LDA vs PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_iris_scaled, y_iris)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA
scatter1 = axes[0].scatter(X_pca_sklearn[:, 0], X_pca_sklearn[:, 1], c=y_iris, cmap='viridis', s=50)
axes[0].set_title('PCA (non supervisé)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# LDA
scatter2 = axes[1].scatter(X_lda[:, 0], X_lda[:, 1], c=y_iris, cmap='viridis', s=50)
axes[1].set_title('LDA (supervisé)')
axes[1].set_xlabel('LD1')
axes[1].set_ylabel('LD2')

plt.suptitle('PCA vs LDA sur Iris', fontsize=14)
plt.tight_layout()
plt.show()

print("LDA sépare mieux les classes car il utilise les labels !")

In [ ]:
# 7.4 LDA comme classifieur
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X_iris_scaled, y_iris, test_size=0.3, random_state=42)

lda_clf = LinearDiscriminantAnalysis()
lda_clf.fit(X_train, y_train)
y_pred = lda_clf.predict(X_test)

print(f"Accuracy LDA comme classifieur : {accuracy_score(y_test, y_pred)*100:.1f}%")

## Partie 8 — Guide de choix

In [ ]:
# Tableau récapitulatif
print("=" * 80)
print("GUIDE DE CHOIX DES MÉTHODES DE RÉDUCTION DE DIMENSION")
print("=" * 80)
print()
print(f"{'Méthode':<15} {'Type':<20} {'Préserve':<25} {'Vitesse':<10} {'Nouveaux pts':<12}")
print("-" * 80)
print(f"{'PCA':<15} {'Linéaire':<20} {'Variance globale':<25} {'⚡⚡⚡':<10} {'✅':<12}")
print(f"{'Kernel PCA':<15} {'Non-linéaire':<20} {'Variance (kernel)':<25} {'⚡⚡':<10} {'✅':<12}")
print(f"{'LDA':<15} {'Linéaire supervisé':<20} {'Séparabilité classes':<25} {'⚡⚡⚡':<10} {'✅':<12}")
print(f"{'t-SNE':<15} {'Non-linéaire':<20} {'Voisinages locaux':<25} {'⚡':<10} {'❌':<12}")
print(f"{'UMAP':<15} {'Non-linéaire':<20} {'Structure locale+globale':<25} {'⚡⚡':<10} {'✅':<12}")
print(f"{'Isomap':<15} {'Non-linéaire':<20} {'Distances géodésiques':<25} {'⚡⚡':<10} {'✅':<12}")
print(f"{'LLE':<15} {'Non-linéaire':<20} {'Relations locales':<25} {'⚡⚡':<10} {'✅':<12}")

In [ ]:
print()
print("=== QUAND UTILISER QUELLE MÉTHODE ? ===")
print()
print("📊 VISUALISATION EXPLORATOIRE")
print("   → Petits datasets (< 10k) : t-SNE")
print("   → Grands datasets : UMAP (plus rapide)")
print()
print("🔧 PRÉ-TRAITEMENT POUR ML")
print("   → Réduction linéaire : PCA")
print("   → Classification : LDA")
print("   → Non-linéaire : Kernel PCA ou UMAP")
print()
print("📖 INTERPRÉTATION DES FEATURES")
print("   → PCA (biplot) ou Sparse PCA")
print()
print("💾 COMPRESSION / DÉBRUITAGE")
print("   → PCA avec n_components choisi par variance expliquée")

## Partie 9 — Cas d'étude complet

In [ ]:
# Fashion MNIST (simplifié)
from sklearn.datasets import fetch_openml

print("Chargement de Fashion MNIST (peut prendre un moment)...")
fashion = fetch_openml('Fashion-MNIST', version=1, as_frame=False, parser='auto')
X_fashion = fashion.data[:5000].astype(float)
y_fashion = fashion.target[:5000].astype(int)

print(f"Shape : {X_fashion.shape} (images 28x28 = 784 pixels)")

# Labels
fashion_labels = ['T-shirt', 'Pantalon', 'Pull', 'Robe', 'Manteau',
                  'Sandale', 'Chemise', 'Basket', 'Sac', 'Bottine']

In [ ]:
# Visualiser quelques images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_fashion[i].reshape(28, 28), cmap='gray')
    ax.set_title(fashion_labels[y_fashion[i]])
    ax.axis('off')
plt.suptitle('Exemples Fashion MNIST')
plt.tight_layout()
plt.show()

In [ ]:
# Pipeline complet
# 1. Normalisation
X_fashion_scaled = X_fashion / 255.0

# 2. PCA pour réduire à 50 composantes
pca_fashion = PCA(n_components=50)
X_pca_50 = pca_fashion.fit_transform(X_fashion_scaled)

print(f"Variance expliquée par 50 composantes : {pca_fashion.explained_variance_ratio_.sum()*100:.1f}%")

# 3. t-SNE sur PCA
print("\nApplication de t-SNE sur les 50 composantes PCA...")
tsne_fashion = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
X_tsne_fashion = tsne_fashion.fit_transform(X_pca_50)

# 4. UMAP direct
print("Application de UMAP...")
umap_fashion = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap_fashion = umap_fashion.fit_transform(X_fashion_scaled)

In [ ]:
# Visualisation finale
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

scatter1 = axes[0].scatter(X_tsne_fashion[:, 0], X_tsne_fashion[:, 1], 
                           c=y_fashion, cmap='tab10', s=10, alpha=0.7)
axes[0].set_title('t-SNE (après PCA 50D)', fontsize=13)
axes[0].set_xticks([])
axes[0].set_yticks([])

scatter2 = axes[1].scatter(X_umap_fashion[:, 0], X_umap_fashion[:, 1], 
                           c=y_fashion, cmap='tab10', s=10, alpha=0.7)
axes[1].set_title('UMAP direct', fontsize=13)
axes[1].set_xticks([])
axes[1].set_yticks([])

# Légende
handles = [plt.Line2D([0], [0], marker='o', color='w', 
                       markerfacecolor=plt.cm.tab10(i/10), markersize=10, label=fashion_labels[i])
           for i in range(10)]
plt.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02, 0.5))

plt.suptitle('Fashion MNIST : t-SNE vs UMAP', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 5-6. Clustering post-réduction + évaluation
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# K-Means sur UMAP
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_umap_fashion)

# Évaluation
ari = adjusted_rand_score(y_fashion, clusters)
print(f"ARI (Adjusted Rand Index) : {ari:.3f}")
print("\n→ Un ARI proche de 1 = bon accord entre clusters et labels réels")
print(f"→ Notre ARI de {ari:.3f} montre que la structure est partiellement capturée")

## Partie 10 — Exercices bonus

In [ ]:
# Bonus 4 : Feature Extraction vs Feature Selection
from sklearn.feature_selection import SelectKBest, f_classif

# Selection
selector = SelectKBest(f_classif, k=2)
X_selected = selector.fit_transform(X_iris_scaled, y_iris)

# Extraction
pca_2 = PCA(n_components=2)
X_extracted = pca_2.fit_transform(X_iris_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_selected[:, 0], X_selected[:, 1], c=y_iris, cmap='viridis', s=50)
axes[0].set_title('Feature Selection (SelectKBest)')
axes[0].set_xlabel(f'Feature {np.where(selector.get_support())[0][0]}')
axes[0].set_ylabel(f'Feature {np.where(selector.get_support())[0][1]}')

axes[1].scatter(X_extracted[:, 0], X_extracted[:, 1], c=y_iris, cmap='viridis', s=50)
axes[1].set_title('Feature Extraction (PCA)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.suptitle('Selection vs Extraction', fontsize=14)
plt.tight_layout()
plt.show()

print("Selection : garde les features originales (interprétable)")
print("Extraction : crée de nouvelles features (combinaisons linéaires)")